In [1]:
from huggingface_hub import hf_hub_download
import pandas as pd

path = hf_hub_download(
    repo_id="project-riz/osu-beatmap-tags",
    filename="tags.csv",  # check exact name on the Files tab
    repo_type="dataset",
)

df = pd.read_csv(path)

/mnt/data/api_project/similar_beatmap/backend/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
entries = []
for _, row in df.iterrows():
    beatmapset_id = row["beatmapset_id"]
    for col in df.columns:
        if col == "beatmapset_id":
            continue
        value = row[col]
        if pd.isna(value) or value == 0:
            continue
        entries.append({
            "beatmapset_id": int(beatmapset_id),
            "tag_name": col,
            "vote_count": int(value)
        })

entries

[{'beatmapset_id': 989587, 'tag_name': 'streams/bursts', 'vote_count': 1},
 {'beatmapset_id': 2594989, 'tag_name': 'streams/bursts', 'vote_count': 3},
 {'beatmapset_id': 2594989, 'tag_name': 'expression/simple', 'vote_count': 3},
 {'beatmapset_id': 2594990, 'tag_name': 'streams/bursts', 'vote_count': 1},
 {'beatmapset_id': 2594991, 'tag_name': 'streams/bursts', 'vote_count': 2},
 {'beatmapset_id': 2594991,
  'tag_name': 'tech/finger control',
  'vote_count': 1},
 {'beatmapset_id': 2594993, 'tag_name': 'skillset/jumps', 'vote_count': 2},
 {'beatmapset_id': 2594993, 'tag_name': 'streams/bursts', 'vote_count': 5},
 {'beatmapset_id': 2594993, 'tag_name': 'jumps/wide', 'vote_count': 5},
 {'beatmapset_id': 4045755, 'tag_name': 'streams/bursts', 'vote_count': 1},
 {'beatmapset_id': 4045755, 'tag_name': 'skillset/streams', 'vote_count': 1},
 {'beatmapset_id': 4045755, 'tag_name': 'streams/flow aim', 'vote_count': 1},
 {'beatmapset_id': 4045755,
  'tag_name': 'tech/finger control',
  'vote_coun

In [ ]:
import psycopg2
from psycopg2.extras import execute_values
from sklearn.feature_extraction.text import CountVectorizer

conn = psycopg2.connect(
    host="localhost",
    dbname="beatmap_similarity",
    user="postgres",
    password=""
)

def insert_batch(conn, rows):
    cols = list(rows[0].keys())
    values = []
    for r in rows:
        row_val = []
        for c in cols:
            row_val.append(r[c])
        values.append(row_val)

    query = f"""
        insert into beatmapset_tags({','.join(cols)})
        values %s
        on conflict (beatmapset_id, tag_name) do nothing
    """

    with conn.cursor() as cur:
        execute_values(cur, query, values)
    conn.commit()


cursor = conn.cursor()

cursor.execute("""
    SELECT beatmapset_id, STRING_AGG(tag_name, ' ') AS tags
    FROM beatmapset_tags
    group by beatmapset_id
""")

rows = cursor.fetchall()

beatmapset_ids = [r[0] for r in rows]
tag_documents = [r[1] for r in rows]

vectorizer = CountVectorizer(tokenizer=lambda x: x.split())
X_sparse = vectorizer.fit_transform(tag_documents)

vocab = vectorizer.vocabulary_
id_to_index = {bms_id: idx for idx, bms_id in enumerate(beatmapset_ids)}

print(vectorizer.vocabulary_)
row_idx = id_to_index[100]
vector_for_100 = X_sparse[row_idx]

{'style/grid': 125, 'snap': 99, 'skillset/precision': 87, 'expression/inis-style': 26, 'expression/old-style': 28, 'revival': 81, 'expression/simple': 32, 'reading/visually': 79, 'dense': 19, 'skillset/streams': 90, 'jumps/linear': 60, 'streams/flow': 108, 'aim': 4, 'style/distance': 118, 'style/messy': 131, 'style/geometric': 124, 'tech/finger': 147, 'control': 16, 'expression/repetition': 31, 'style/symmetrical': 137, 'reading/overlaps': 77, 'additions/custom': 1, 'skin': 93, 'expression/difficulty': 23, 'spike': 101, 'jumps/cross-screen': 58, 'jumps/squares': 62, 'skillset/jumps': 86, 'sliders/complex': 95, 'slidershapes': 98, 'style/freeform': 122, 'streams/bursts': 105, 'style/clean': 116, 'style/hexgrid': 127, 'jumps/wide': 66, 'expression/improvisation': 25, 'gimmick/ninja': 44, 'spinners': 102, 'streams/cutstreams': 106, 'streams/stamina': 112, 'style/avant-garde': 113, 'meta/variable': 73, 'timing': 155, 'sliders/high': 96, 'sv': 142, 'gimmick/circle': 37, 'only': 76, 'express

/mnt/data/api_project/similar_beatmap/backend/.venv/lib/python3.12/site-packages/sklearn/feature_extraction/text.py:527: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
